# Merge IPC Food Insecurity Data with Rainfall Features

This notebook combines the cleaned IPC county-level food insecurity dataset with county-level CHIRPS rainfall features.

Main tasks:
- Load the processed IPC dataset
- Load the processed rainfall feature dataset
- Align rainfall months with IPC analysis periods
- Join rainfall predictors to IPC outcomes
- Save a merged modeling-ready dataset

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
IPC_PATH = Path("../02_data/processed/ipc_max_phase_per_county.csv")
RAINFALL_FEATURES_PATH = Path("../02_data/processed/county_rainfall_features.csv")

ipc = pd.read_csv(IPC_PATH)
rainfall = pd.read_csv(RAINFALL_FEATURES_PATH)

print("IPC shape:", ipc.shape)
print("Rainfall shape:", rainfall.shape)

ipc.head()

IPC shape: (322, 7)
Rainfall shape: (1978, 10)


,date,analysis_period,county,max_ipc_phase,phase_3_plus_population,total_population,phase_3_plus_percentage
0,2019-07-01,Jul 2019,Baringo,4,105555,703697,0.15
1,2019-07-01,Jul 2019,Embu,3,32883,219220,0.15
2,2019-07-01,Jul 2019,Garissa,4,151183,431950,0.35
3,2019-07-01,Jul 2019,Isiolo,4,54413,155465,0.35
4,2019-07-01,Jul 2019,Kajiado,3,43536,870721,0.05


In [3]:
rainfall.head()

,date,year,month,county,mean_rainfall_mm,source_file,rainfall_3_month_total,rainfall_6_month_total,rainfall_3_month_avg,rainfall_6_month_avg
0,2019-01-01,2019,1,Baringo,7.184619,chirps-v2.0.2019.01.tif,7.184619,7.184619,7.184619,7.184619
1,2019-02-01,2019,2,Baringo,4.409290,chirps-v2.0.2019.02.tif,11.593909,11.593909,5.796955,5.796955
2,2019-03-01,2019,3,Baringo,17.688416,chirps-v2.0.2019.03.tif,29.282325,29.282325,9.760775,9.760775
3,2019-04-01,2019,4,Baringo,40.918780,chirps-v2.0.2019.04.tif,63.016487,70.201106,21.005496,17.550277
4,2019-05-01,2019,5,Baringo,29.830008,chirps-v2.0.2019.05.tif,88.437204,100.031114,29.479068,20.006223


In [4]:
ipc["date"] = pd.to_datetime(ipc["date"])
rainfall["date"] = pd.to_datetime(rainfall["date"])

print(ipc["date"].min(), ipc["date"].max())
print(rainfall["date"].min(), rainfall["date"].max())

2019-07-01 00:00:00 2026-02-01 00:00:00
2019-01-01 00:00:00 2026-02-01 00:00:00


## Align Rainfall Features with IPC Analysis Periods

The IPC dataset is reported at specific analysis periods, while rainfall is monthly.

To avoid using rainfall from the same month as the IPC outcome, this notebook joins each IPC period with rainfall features from the previous month.

For example:
- IPC period `Jul 2019` uses rainfall features ending in `Jun 2019`
- IPC period `Feb 2020` uses rainfall features ending in `Jan 2020`

This makes the rainfall variables more suitable as predictors.

In [5]:
ipc["rainfall_date"] = ipc["date"] - pd.DateOffset(months=1)

ipc[["date", "analysis_period", "rainfall_date", "county"]].head(10)

,date,analysis_period,rainfall_date,county
0,2019-07-01,Jul 2019,2019-06-01,Baringo
1,2019-07-01,Jul 2019,2019-06-01,Embu
2,2019-07-01,Jul 2019,2019-06-01,Garissa
3,2019-07-01,Jul 2019,2019-06-01,Isiolo
4,2019-07-01,Jul 2019,2019-06-01,Kajiado
5,2019-07-01,Jul 2019,2019-06-01,Kilifi
6,2019-07-01,Jul 2019,2019-06-01,Kitui
7,2019-07-01,Jul 2019,2019-06-01,Kwale
8,2019-07-01,Jul 2019,2019-06-01,Laikipia
9,2019-07-01,Jul 2019,2019-06-01,Lamu


In [6]:
merged = ipc.merge(
    rainfall,
    left_on=["county", "rainfall_date"],
    right_on=["county", "date"],
    how="left",
    suffixes=("_ipc", "_rainfall")
)

print("Merged shape:", merged.shape)
merged.head()

Merged shape: (322, 17)


,date_ipc,analysis_period,county,max_ipc_phase,phase_3_plus_population,total_population,phase_3_plus_percentage,rainfall_date,date_rainfall,year,month,mean_rainfall_mm,source_file,rainfall_3_month_total,rainfall_6_month_total,rainfall_3_month_avg,rainfall_6_month_avg
0,2019-07-01,Jul 2019,Baringo,4,105555,703697,0.15,2019-06-01,2019-06-01,2019,6,72.692665,chirps-v2.0.2019.06.tif,143.441454,172.723779,47.813818,28.787296
1,2019-07-01,Jul 2019,Embu,3,32883,219220,0.15,2019-06-01,2019-06-01,2019,6,8.068131,chirps-v2.0.2019.06.tif,123.946965,156.388124,41.315655,26.064687
2,2019-07-01,Jul 2019,Garissa,4,151183,431950,0.35,2019-06-01,2019-06-01,2019,6,5.417682,chirps-v2.0.2019.06.tif,55.185197,66.812339,18.395066,11.135390
3,2019-07-01,Jul 2019,Isiolo,4,54413,155465,0.35,2019-06-01,2019-06-01,2019,6,0.936229,chirps-v2.0.2019.06.tif,26.591658,40.062429,8.863886,6.677072
4,2019-07-01,Jul 2019,Kajiado,3,43536,870721,0.05,2019-06-01,2019-06-01,2019,6,6.391582,chirps-v2.0.2019.06.tif,62.716861,88.344720,20.905620,14.724120


## Validate the Merged Dataset

After joining IPC records with rainfall features, the next step is to check whether any IPC records failed to match with rainfall data.

If the merge worked correctly, the rainfall feature columns should not contain missing values.

In [7]:
rainfall_columns = [
    "mean_rainfall_mm",
    "rainfall_3_month_total",
    "rainfall_6_month_total",
    "rainfall_3_month_avg",
    "rainfall_6_month_avg"
]

merged[rainfall_columns].isna().sum()

mean_rainfall_mm          0
rainfall_3_month_total    0
rainfall_6_month_total    0
rainfall_3_month_avg      0
rainfall_6_month_avg      0
dtype: int64

In [8]:
merged[["date_ipc", "analysis_period", "county", "rainfall_date", "mean_rainfall_mm", "rainfall_3_month_total", "rainfall_6_month_total"]].head(10)

,date_ipc,analysis_period,county,rainfall_date,mean_rainfall_mm,rainfall_3_month_total,rainfall_6_month_total
0,2019-07-01,Jul 2019,Baringo,2019-06-01,72.692665,143.441454,172.723779
1,2019-07-01,Jul 2019,Embu,2019-06-01,8.068131,123.946965,156.388124
2,2019-07-01,Jul 2019,Garissa,2019-06-01,5.417682,55.185197,66.812339
3,2019-07-01,Jul 2019,Isiolo,2019-06-01,0.936229,26.591658,40.062429
4,2019-07-01,Jul 2019,Kajiado,2019-06-01,6.391582,62.716861,88.344720
5,2019-07-01,Jul 2019,Kilifi,2019-06-01,33.560910,202.082382,227.631483
6,2019-07-01,Jul 2019,Kitui,2019-06-01,2.934999,65.223084,89.192118
7,2019-07-01,Jul 2019,Kwale,2019-06-01,27.190882,221.532774,249.557383
8,2019-07-01,Jul 2019,Laikipia,2019-06-01,53.306343,131.485222,167.249873
9,2019-07-01,Jul 2019,Lamu,2019-06-01,26.783705,174.986271,188.545978


## Create Final IPC-Rainfall Modeling Dataset

The merge successfully joined IPC food insecurity records with rainfall features from the previous month.

The final dataset keeps the main IPC outcome variables and rainfall predictor variables. This creates a modeling-ready table for exploring the relationship between rainfall conditions and food insecurity severity.

In [9]:
final_model_df = merged[
    [
        "date_ipc",
        "analysis_period",
        "county",
        "max_ipc_phase",
        "phase_3_plus_population",
        "total_population",
        "phase_3_plus_percentage",
        "rainfall_date",
        "mean_rainfall_mm",
        "rainfall_3_month_total",
        "rainfall_6_month_total",
        "rainfall_3_month_avg",
        "rainfall_6_month_avg"
    ]
].copy()

final_model_df = final_model_df.rename(columns={
    "date_ipc": "ipc_date"
})

print("Final modeling dataset shape:", final_model_df.shape)
final_model_df.head(10)

Final modeling dataset shape: (322, 13)


,ipc_date,analysis_period,county,max_ipc_phase,phase_3_plus_population,total_population,phase_3_plus_percentage,rainfall_date,mean_rainfall_mm,rainfall_3_month_total,rainfall_6_month_total,rainfall_3_month_avg,rainfall_6_month_avg
0,2019-07-01,Jul 2019,Baringo,4,105555,703697,0.15,2019-06-01,72.692665,143.441454,172.723779,47.813818,28.787296
1,2019-07-01,Jul 2019,Embu,3,32883,219220,0.15,2019-06-01,8.068131,123.946965,156.388124,41.315655,26.064687
2,2019-07-01,Jul 2019,Garissa,4,151183,431950,0.35,2019-06-01,5.417682,55.185197,66.812339,18.395066,11.135390
3,2019-07-01,Jul 2019,Isiolo,4,54413,155465,0.35,2019-06-01,0.936229,26.591658,40.062429,8.863886,6.677072
4,2019-07-01,Jul 2019,Kajiado,3,43536,870721,0.05,2019-06-01,6.391582,62.716861,88.344720,20.905620,14.724120
5,2019-07-01,Jul 2019,Kilifi,3,209996,1399975,0.15,2019-06-01,33.560910,202.082382,227.631483,67.360794,37.938580
6,2019-07-01,Jul 2019,Kitui,3,219537,1097687,0.20,2019-06-01,2.934999,65.223084,89.192118,21.741028,14.865353
7,2019-07-01,Jul 2019,Kwale,3,123030,820199,0.15,2019-06-01,27.190882,221.532774,249.557383,73.844258,41.592897
8,2019-07-01,Jul 2019,Laikipia,3,50571,505712,0.10,2019-06-01,53.306343,131.485222,167.249873,43.828407,27.874979
9,2019-07-01,Jul 2019,Lamu,3,25629,128144,0.20,2019-06-01,26.783705,174.986271,188.545978,58.328757,31.424330


## Save Final IPC-Rainfall Modeling Dataset

The IPC food insecurity dataset has been successfully merged with rainfall features from the previous month.

The final dataset contains one row per county per IPC analysis period. It includes both food insecurity outcome variables and rainfall predictor variables.

This file will be used for the next stage of analysis, where rainfall patterns will be compared with IPC food insecurity severity.

In [10]:
FINAL_MODEL_PATH = Path("../02_data/processed/ipc_rainfall_modeling_dataset.csv")

final_model_df.to_csv(FINAL_MODEL_PATH, index=False)

print(f"Saved final modeling dataset to: {FINAL_MODEL_PATH}")

Saved final modeling dataset to: ..\02_data\processed\ipc_rainfall_modeling_dataset.csv


In [11]:
FINAL_MODEL_PATH.exists()

True

## Merge Result

The final IPC-rainfall modeling dataset was successfully created and saved as `02_data/processed/ipc_rainfall_modeling_dataset.csv`.

The dataset contains 322 rows and 13 columns. Each row represents one county in one IPC analysis period, with rainfall features from the previous month attached.

This completes the first merged dataset needed to analyze the relationship between rainfall conditions and food insecurity severity.